# 🔧 Plantilla interactiva: Machine Learning para Mantenimiento Predictivo

Esta plantilla te permite **entrenar y evaluar modelos de Machine Learning** para predecir fallas de maquinaria, sin escribir código adicional. Solo tienes que ejecutar las celdas en orden y usar los controles interactivos.

**Flujo de trabajo:**
1. Cargar datos (usa el dataset de ejemplo incluido o sube el tuyo)
2. Explorar el dataset
3. Elegir la variable objetivo (target) y las variables predictoras (features)
4. Definir el porcentaje de datos de entrenamiento / prueba
5. Elegir un modelo de Machine Learning sugerido
6. Entrenar y ver una retrospectiva de aceptación (métricas de desempeño)

---

### 📦 Dataset de ejemplo incluido

Se incluye el **AI4I 2020 Predictive Maintenance Dataset** (UCI Machine Learning Repository), un dataset sintético que imita datos reales de una planta de manufactura (fresadora industrial). Contiene 10,000 registros con variables de sensores (temperatura, velocidad de rotación, torque, desgaste de herramienta) y una etiqueta de si la máquina falló o no.

> Fuente: Matzka, S. (2020). *AI4I 2020 Predictive Maintenance Dataset*. UCI Machine Learning Repository. https://doi.org/10.24432/C5HS5C (licencia CC BY 4.0)

Si quieres usar tus propios datos, simplemente sube tu archivo CSV en la celda de carga de datos y sigue el mismo flujo.


## 1. Preparar el entorno
Ejecuta esta celda primero para instalar (si hace falta) e importar todas las librerías necesarias.

In [ ]:
# Si alguna librería falta, descomenta la siguiente línea:
# %pip install pandas numpy scikit-learn matplotlib seaborn ipywidgets xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Librerías cargadas correctamente ✅")


## 2. Cargar el dataset

Puedes:
- **Usar el dataset de ejemplo** (AI4I 2020) tal cual, o
- **Subir tu propio CSV** con el botón de abajo (si subes un archivo, este reemplaza al de ejemplo).


In [ ]:
DEFAULT_DATA_PATH = "data/ai4i2020.csv"

upload_widget = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Subir CSV"
)

output_carga = widgets.Output()

df = None

def cargar_datos(_=None):
    global df
    with output_carga:
        clear_output()
        if len(upload_widget.value) > 0:
            uploaded_file = list(upload_widget.value.values())[0]
            import io
            df = pd.read_csv(io.BytesIO(uploaded_file['content']))
            print(f"✅ Dataset propio cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
        else:
            df = pd.read_csv(DEFAULT_DATA_PATH)
            print(f"✅ Dataset de ejemplo (AI4I 2020) cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
        display(df.head())

boton_cargar = widgets.Button(description="Cargar datos", button_style="primary")
boton_cargar.on_click(cargar_datos)

display(widgets.HTML("<b>Opcional:</b> sube tu propio CSV (si no subes nada, se usará el dataset de ejemplo)"))
display(upload_widget)
display(boton_cargar)
display(output_carga)

# Carga automática inicial con el dataset de ejemplo
cargar_datos()


## 3. Exploración rápida de los datos
Revisa tipos de datos, valores nulos y estadísticas básicas antes de continuar.

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nEstadísticas descriptivas:")
display(df.describe(include='all').T)


In [ ]:
# Distribución de columnas numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) > 0:
    df[num_cols].hist(figsize=(14, 10), bins=30)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron columnas numéricas.")


## 4. Selecciona la variable objetivo (target) y las variables predictoras (features)

- **Target**: lo que quieres predecir (ej. si la máquina falla o no).
- **Features**: las variables que usará el modelo para predecir (ej. temperatura, velocidad, torque, etc.).

> Tip: para el dataset de ejemplo, usa **`Target`** (0 = no falla, 1 = falla) como variable objetivo, y evita usar `Failure Type` como feature porque revela directamente el resultado (fuga de información / *data leakage*).


In [ ]:
target_dropdown = widgets.Dropdown(
    options=df.columns.tolist(),
    description="Target:",
    value=("Target" if "Target" in df.columns else df.columns[-1]),
    style={'description_width': 'initial'}
)

features_select = widgets.SelectMultiple(
    options=[c for c in df.columns if c != target_dropdown.value],
    description="Features:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="400px", height="150px")
)

# Preselección razonable para el dataset de ejemplo
sugeridas_default = [c for c in ["Air temperature [K]", "Process temperature [K]",
                                  "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]", "Type"]
                      if c in features_select.options]
if sugeridas_default:
    features_select.value = tuple(sugeridas_default)

def actualizar_features(change):
    features_select.options = [c for c in df.columns if c != target_dropdown.value]

target_dropdown.observe(actualizar_features, names='value')

display(target_dropdown)
display(widgets.HTML("<i>Mantén Ctrl (o Cmd en Mac) presionado para seleccionar varias features</i>"))
display(features_select)


## 5. Tipo de problema y % de datos para entrenamiento

El notebook detecta automáticamente si el problema es de **clasificación** (target con pocas categorías, ej. falla/no falla) o de **regresión** (target numérico continuo, ej. vida útil restante).

También define aquí qué porcentaje de tus datos se usará para **entrenar** el modelo y qué porcentaje se reserva para **probarlo** (evaluar qué tan bien generaliza a datos que no ha visto).


In [ ]:
def detectar_tipo_problema(serie):
    if serie.dtype == object or serie.nunique() <= 10:
        return "clasificacion"
    return "regresion"

tipo_problema = detectar_tipo_problema(df[target_dropdown.value])
print(f"Tipo de problema detectado: {tipo_problema.upper()}")
if tipo_problema == "clasificacion":
    print("Distribución de clases del target:")
    print(df[target_dropdown.value].value_counts())


In [ ]:
split_slider = widgets.IntSlider(
    value=80, min=50, max=95, step=5,
    description="% Entrenamiento:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="400px")
)

display(split_slider)
display(widgets.HTML("<i>El resto de los datos (100% - % entrenamiento) se usará como conjunto de prueba (test)</i>"))


## 6. Modelos de Machine Learning sugeridos

Según el tipo de problema detectado, estos son los modelos disponibles. Elige el que quieras probar (puedes volver a esta celda y repetir con otro modelo para comparar resultados).


In [ ]:
MODELOS_CLASIFICACION = {
    "Regresión Logística": LogisticRegression(max_iter=1000),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM (Máquina de Vectores de Soporte)": SVC(probability=True, random_state=42),
    "K-Vecinos Cercanos (KNN)": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
}

MODELOS_REGRESION = {
    "Regresión Lineal": LinearRegression(),
    "Árbol de Decisión": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=200),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVM (SVR)": SVR(),
    "K-Vecinos Cercanos (KNN)": KNeighborsRegressor(),
}

modelos_disponibles = MODELOS_CLASIFICACION if tipo_problema == "clasificacion" else MODELOS_REGRESION

modelo_dropdown = widgets.Dropdown(
    options=list(modelos_disponibles.keys()),
    description="Modelo:",
    style={'description_width': 'initial'}
)

display(widgets.HTML(f"<b>Problema tipo:</b> {tipo_problema}"))
display(modelo_dropdown)


## 7. Entrenar el modelo

Al presionar el botón:
1. Se preparan los datos (codificación de variables categóricas, escalado).
2. Se dividen en entrenamiento/prueba según el % elegido.
3. Se entrena el modelo seleccionado.
4. Se muestra la **retrospectiva de aceptación** (métricas de desempeño).


In [ ]:
output_entrenamiento = widgets.Output()
resultados = {}

def preparar_datos(features, target_col):
    X = df[list(features)].copy()
    y = df[target_col].copy()

    # Codificar columnas categóricas de X
    encoders = {}
    for col in X.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

    # Codificar target si es categórico
    target_encoder = None
    if y.dtype == object:
        target_encoder = LabelEncoder()
        y = target_encoder.fit_transform(y.astype(str))

    return X, y, encoders, target_encoder

def entrenar_modelo(_=None):
    with output_entrenamiento:
        clear_output()

        features = features_select.value
        target_col = target_dropdown.value

        if len(features) == 0:
            print("⚠️ Selecciona al menos una feature en la celda de arriba.")
            return

        X, y, encoders, target_encoder = preparar_datos(features, target_col)

        train_pct = split_slider.value / 100
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, train_size=train_pct, random_state=42,
            stratify=y if tipo_problema == "clasificacion" and pd.Series(y).nunique() > 1 else None
        )

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        modelo = modelos_disponibles[modelo_dropdown.value]
        modelo.fit(X_train_s, y_train)
        y_pred = modelo.predict(X_test_s)

        print(f"Modelo: {modelo_dropdown.value}")
        print(f"Datos de entrenamiento: {len(X_train)} filas ({split_slider.value}%)")
        print(f"Datos de prueba: {len(X_test)} filas ({100 - split_slider.value}%)")
        print("-" * 60)

        if tipo_problema == "clasificacion":
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

            print(f"✅ Accuracy:  {acc:.4f}")
            print(f"✅ Precision: {prec:.4f}")
            print(f"✅ Recall:    {rec:.4f}")
            print(f"✅ F1-score:  {f1:.4f}")
            print("\nReporte de clasificación completo:")
            print(classification_report(y_test, y_pred, zero_division=0))

            fig, ax = plt.subplots(figsize=(5, 5))
            cm = confusion_matrix(y_test, y_pred)
            ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
            ax.set_title("Matriz de confusión")
            plt.show()

            resultados['metricas'] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
        else:
            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            print(f"✅ MAE:  {mae:.4f}")
            print(f"✅ RMSE: {rmse:.4f}")
            print(f"✅ R²:   {r2:.4f}")

            fig, ax = plt.subplots(figsize=(6, 5))
            ax.scatter(y_test, y_pred, alpha=0.5)
            lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
            ax.plot(lims, lims, 'r--', label="Predicción perfecta")
            ax.set_xlabel("Valor real")
            ax.set_ylabel("Valor predicho")
            ax.set_title("Real vs. Predicho")
            ax.legend()
            plt.show()

            resultados['metricas'] = {'mae': mae, 'rmse': rmse, 'r2': r2}

        # Importancia de variables (si el modelo lo soporta)
        if hasattr(modelo, "feature_importances_"):
            importancias = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=False)
            fig, ax = plt.subplots(figsize=(8, 4))
            importancias.plot(kind="bar", ax=ax, color="steelblue")
            ax.set_title("Importancia de variables")
            ax.set_ylabel("Importancia")
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        elif hasattr(modelo, "coef_"):
            coefs = pd.Series(np.ravel(modelo.coef_), index=features).sort_values(key=abs, ascending=False)
            fig, ax = plt.subplots(figsize=(8, 4))
            coefs.plot(kind="bar", ax=ax, color="steelblue")
            ax.set_title("Coeficientes del modelo (importancia relativa)")
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()

        resultados['modelo'] = modelo
        resultados['scaler'] = scaler
        print("\n✅ Entrenamiento completado. Puedes cambiar el modelo o el % de split arriba y volver a entrenar para comparar.")

boton_entrenar = widgets.Button(description="🚀 Entrenar modelo", button_style="success")
boton_entrenar.on_click(entrenar_modelo)

display(boton_entrenar)
display(output_entrenamiento)


## 8. Notas y próximos pasos

- **Compara modelos**: cambia el modelo en la celda 6 y vuelve a presionar "Entrenar modelo" para comparar métricas.
- **Experimenta con el split**: prueba distintos porcentajes de entrenamiento/prueba (ej. 70/30 vs 80/20) y observa cómo cambian las métricas.
- **Cuidado con el *data leakage***: si usas el dataset de ejemplo, no incluyas `Failure Type` como feature, ya que revela directamente si hubo falla.
- **Dataset desbalanceado**: en el dataset de ejemplo, las fallas son solo ~3.4% de los casos. Un accuracy alto puede ser engañoso; presta atención a *precision*, *recall* y *f1-score*, especialmente para la clase minoritaria (falla = 1).
- **Siguiente nivel**: se puede añadir validación cruzada (`cross_val_score`), búsqueda de hiperparámetros (`GridSearchCV`) o técnicas de balanceo de clases (`SMOTE`) como extensión de esta plantilla.
